In [1]:
import os

print(os.listdir())

['.config', 'sample_data']


In [3]:
import pandas as pd
import glob

for file in glob.glob("*.csv"):
    data = pd.read_csv(file)
    print("\nFile:", file)
    print("Shape:", data.shape)
    print("Columns:", list(data.columns))

In [4]:
import os
print(os.listdir())

['.config', 'ecWvPsdV2FNmRZAa.csv', 'jbFzPdd2gB2Ex2TY.csv', 'qDCCmpyfss1kBJJB.csv', 'sample_data']


In [5]:
import pandas as pd
import glob

for file in glob.glob("*.csv"):
    data = pd.read_csv(file)
    print("\nFile:", file)
    print("Shape:", data.shape)
    print("Columns:", list(data.columns))


File: ecWvPsdV2FNmRZAa.csv
Shape: (30, 3)
Columns: ['note_id', 'appointment_id', 'note_text']

File: jbFzPdd2gB2Ex2TY.csv
Shape: (8, 6)
Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

File: qDCCmpyfss1kBJJB.csv
Shape: (1020, 14)
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']


In [6]:
notes_df = pd.read_csv("ecWvPsdV2FNmRZAa.csv")

print("Follow-up notes loaded successfully!")
print("Shape:", notes_df.shape)
display(notes_df.head(10))

Follow-up notes loaded successfully!
Shape: (30, 3)


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [7]:
notes_to_analyse = notes_df.head(10).copy()

# Remove blank notes
notes_to_analyse = notes_to_analyse.dropna(
    subset=["note_text"]
)

print("Notes selected:", len(notes_df.head(10)))
print("Notes to analyse:", len(notes_to_analyse))
print("Skipped blank notes:", len(notes_df.head(10)) - len(notes_to_analyse))

display(notes_to_analyse)

Notes selected: 10
Notes to analyse: 9
Skipped blank notes: 1


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [8]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [10]:
!pip install -q google-genai==2.12.1 google-auth==2.49.0

In [11]:
import google.genai

print("Gemini library installed successfully!")
print("Version:", google.genai.__version__)

Gemini library installed successfully!
Version: 2.12.1


In [12]:
from google.colab import userdata
from google import genai

# Retrieve API key securely
api_key = userdata.get("GEMINI_API_KEY")

# Connect to Gemini
client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [13]:
from pydantic import BaseModel, Field
from typing import Literal

class NoteClassification(BaseModel):
    reason_category: Literal[
        "Reminder Issue",
        "Travel Issue",
        "Cost Concern",
        "Medical Issue",
        "Work Conflict",
        "Forgot",
        "Other"
    ]

    sentiment: Literal[
        "Negative",
        "Neutral",
        "Positive"
    ]

    urgency: Literal[
        "Low",
        "Medium",
        "High"
    ]

    summary: str = Field(
        description="A short summary of the patient's reason for missing the appointment."
    )

print("Pydantic classification structure created!")

Pydantic classification structure created!


In [14]:
import json
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

# Allowed root causes
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Pydantic validation model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

print("Correct Pydantic validation model created!")
print("Allowed root causes:", ALLOWED_ROOT_CAUSES)

Correct Pydantic validation model created!
Allowed root causes: ['forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule', 'other']


In [27]:
from google import genai
from google.genai import types
from google.colab import userdata
import json

MODEL_NAME = "gemini-3.5-flash"

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You classify clinic call-centre notes about missed appointments.
Return only the requested JSON object.
Do not invent details that are not present in the note.
"""

api_call_count = 0

def analyse_note(note_text):
    global api_call_count

    prompt = f"""
Context: You are analysing a clinic call-centre note about a missed appointment.

Role: Act as a careful healthcare operations analyst.

Action:
Choose exactly one root_cause from this list:
{ALLOWED_ROOT_CAUSES}

If the evidence is unclear or does not fit any category, choose "other".
Estimate confidence from 0 to 1.
Write a summary of no more than 25 words.
Recommend one practical action for the clinic.
Do not guess or invent facts.

Tone: Neutral, concise, and evidence-based.

Format: Return only a JSON object with exactly these four fields:
root_cause, confidence, summary, recommended_action.
The root_cause must be a plain string.

Note: {note_text}
"""

    api_call_count += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)


print("Gemini analysis function created successfully!")

Gemini analysis function created successfully!


In [28]:
n001_text = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

n001_result = analyse_note(n001_text)

print("Original note:", n001_text)
print("\nGemini's analysis:")
print(n001_result)

print("\nAPI calls so far:", api_call_count)

Original note: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Gemini's analysis:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient forgot her appointment after not receiving a reminder message and requested to rebook for next week.', 'recommended_action': 'Reschedule the patient for next week and verify contact details and automated reminder system settings.'}

API calls so far: 1


In [16]:
# Select the first note (N001)
n001_text = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

# Send the note to Gemini
n001_result = analyse_note(n001_text)

# Display the result
print("Original note:", n001_text)
print("\nGemini's analysis:")
print(n001_result)

print("\nAPI calls so far:", api_call_count)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [18]:
n001_text = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

n001_result = analyse_note(n001_text)

print("Original note:", n001_text)
print("\nGemini's analysis:")
print(n001_result)

print("\nAPI calls so far:", api_call_count)

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Thinking level MINIMAL is not supported for this model. Please retry with other thinking level.', 'status': 'INVALID_ARGUMENT'}}

In [23]:
n001_text = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

n001_result = analyse_note(n001_text)

print("Original note:", n001_text)
print("\nGemini's analysis:")
print(n001_result)

print("\nAPI calls so far:", api_call_count)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [24]:
import time

print("Waiting 30 seconds before retrying...")
time.sleep(30)

print("Ready to retry!")

Waiting 30 seconds before retrying...
Ready to retry!


In [25]:
n001_text = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

n001_result = analyse_note(n001_text)

print("Original note:", n001_text)
print("\nGemini's analysis:")
print(n001_result)

print("\nAPI calls so far:", api_call_count)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [26]:
for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [29]:
import time

# Select the first 10 notes
notes_to_analyse = notes_df.head(10).copy()

print("Total notes selected:", len(notes_to_analyse))

# Count blank notes
blank_count = notes_to_analyse["note_text"].isna().sum()

print("Blank notes:", blank_count)
print("Notes to send to Gemini:", len(notes_to_analyse) - blank_count)

Total notes selected: 10
Blank notes: 1
Notes to send to Gemini: 9


In [30]:
results = []

for index, row in notes_to_analyse.iterrows():

    note_text = row["note_text"]

    # Prepare common row information
    result_row = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Skip missing or blank notes
    if pd.isna(note_text) or str(note_text).strip() == "":
        result_row.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

        results.append(result_row)
        print(row["note_id"], "- Blank note skipped")
        continue

    try:
        # Send note to Gemini
        prompt_result = analyse_note(note_text)

        # Preserve the returned fields
        result_row["root_cause"] = prompt_result.get("root_cause", "")
        result_row["confidence"] = prompt_result.get("confidence", "")
        result_row["summary"] = prompt_result.get("summary", "")
        result_row["recommended_action"] = prompt_result.get(
            "recommended_action", ""
        )

        # Validate with Pydantic
        validated = FollowupInsight(**prompt_result)

        # Check allowed root causes separately
        if validated.root_cause not in ALLOWED_ROOT_CAUSES:
            raise ValueError("Root cause is not in the allowed list")

        result_row["validation_status"] = "valid"

    except Exception as e:
        result_row["validation_status"] = "invalid"
        print(row["note_id"], "- Validation/API error:", str(e))

    results.append(result_row)

    print(row["note_id"], "-", result_row["validation_status"])

    # Wait 13 seconds between API calls
    time.sleep(13)

# Convert results into a DataFrame
analysed_df = pd.DataFrame(results)

print("\nAnalysis completed!")
print("Total records:", len(analysed_df))
print("API calls so far:", api_call_count)

N001 - valid
N002 - valid
N003 - valid
N004 - Blank note skipped
N005 - valid
N006 - valid
N007 - valid
N008 - valid
N009 - valid
N010 - valid

Analysis completed!
Total records: 10
API calls so far: 10


In [31]:
def analyse_note_raw(note_text):
    global api_call_count

    prompt = f"""
Context: You are analysing a clinic call-centre note about a missed appointment.

Role: Act as a careful healthcare operations analyst.

Action:
Choose exactly one root_cause from this list:
{ALLOWED_ROOT_CAUSES}

If evidence is unclear, choose "other".
Estimate confidence from 0 to 1.
Write a summary of no more than 25 words.
Recommend one practical action.
Do not invent facts.

Tone: Neutral, concise, evidence-based.

Format: Return only JSON with:
root_cause, confidence, summary, recommended_action.

Note: {note_text}
"""

    api_call_count += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw_text = response.text
    parsed_data = json.loads(raw_text)

    return raw_text, parsed_data


print("Raw response capture function created!")

Raw response capture function created!


In [33]:
results = []

for index, row in notes_to_analyse.iterrows():

    note_text = row["note_text"]

    result_row = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Skip blank notes
    if pd.isna(note_text) or str(note_text).strip() == "":
        result_row.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

        results.append(result_row)
        print(row["note_id"], "- Blank note skipped")
        continue

    try:
        # Get the exact response text from Gemini
        raw_text = generate_raw_response(note_text)
        result_row["raw_response"] = raw_text

        # Convert JSON text into a dictionary
        prompt_result = json.loads(raw_text)

        # Copy the returned fields
        result_row["root_cause"] = prompt_result.get("root_cause", "")
        result_row["confidence"] = prompt_result.get("confidence", "")
        result_row["summary"] = prompt_result.get("summary", "")
        result_row["recommended_action"] = prompt_result.get(
            "recommended_action", ""
        )

        # Pydantic validation
        validated = FollowupInsight(**prompt_result)

        if validated.root_cause not in ALLOWED_ROOT_CAUSES:
            raise ValueError("Root cause is not in the allowed list")

        result_row["validation_status"] = "valid"

    except Exception as e:
        result_row["validation_status"] = "invalid"
        print(row["note_id"], "- Error:", str(e))

    results.append(result_row)
    print(row["note_id"], "-", result_row["validation_status"])

    time.sleep(13)

analysed_df = pd.DataFrame(results)

print("\nTotal records:", len(analysed_df))
print("API calls so far:", api_call_count)

N001 - valid
N002 - valid
N003 - valid
N004 - Blank note skipped
N005 - valid
N006 - valid
N007 - valid
N008 - valid
N009 - valid
N010 - valid

Total records: 10
API calls so far: 19


In [32]:
def generate_raw_response(note_text):
    global api_call_count

    prompt = f"""
Context: You are analysing a clinic call-centre note about a missed appointment.
Role: Act as a careful healthcare operations analyst.
Action: Choose exactly one root_cause from {ALLOWED_ROOT_CAUSES}.
If evidence is unclear, choose other. Estimate confidence from 0 to 1.
Write a summary of no more than 25 words and recommend one practical action.
Do not invent facts.
Tone: Neutral, concise, evidence-based.
Format: Return only JSON with root_cause, confidence, summary, recommended_action.
Note: {note_text}
"""

    api_call_count += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return response.text

print("Raw response helper ready!")

Raw response helper ready!


In [34]:
print(analysed_df[[
    "note_id", "root_cause", "confidence", "validation_status"
]])

print("\nRaw response lengths:")
print(analysed_df["raw_response"].str.len())

analysed_df.to_csv("notes_analysed.csv", index=False)

print("\nnotes_analysed.csv saved successfully!")

  note_id             root_cause  confidence validation_status
0    N001  forgot_or_no_reminder        0.95             valid
1    N002  distance_or_transport        0.95             valid
2    N003            cost_or_fee        0.95             valid
3    N004                  other        0.00     skipped_blank
4    N005   illness_or_emergency        0.95             valid
5    N006       work_or_schedule        0.95             valid
6    N007  forgot_or_no_reminder        0.95             valid
7    N008  distance_or_transport        0.95             valid
8    N009                  other        0.85             valid
9    N010       work_or_schedule        0.95             valid

Raw response lengths:
0    295
1    289
2    274
3      0
4    277
5    287
6    270
7    306
8    237
9    242
Name: raw_response, dtype: int64

notes_analysed.csv saved successfully!


In [35]:
bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Test summary",
    "recommended_action": "Test action"
}

bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.5,
    "summary": "Test summary"
}

for label, bad_reply in [
    ("Bad reply 1: Invalid confidence", bad_reply_1),
    ("Bad reply 2: Missing action", bad_reply_2)
]:
    try:
        FollowupInsight(**bad_reply)
        print(label, "- Unexpectedly passed")
    except ValidationError as e:
        print(label, "- ValidationError caught!")
        print(e)

Bad reply 1: Invalid confidence - ValidationError caught!
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
Bad reply 2: Missing action - ValidationError caught!
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...ummary': 'Test summary'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [36]:
comparison_notes = notes_to_analyse[
    notes_to_analyse["note_id"].isin(["N007", "N009"])
]

comparison_results = []

for _, row in comparison_notes.iterrows():
    note_text = row["note_text"]

    prompt = f"Classify this note: {note_text}"

    api_call_count += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    v1_data = json.loads(response.text)

    v2_data = analysed_df.loc[
        analysed_df["note_id"] == row["note_id"]
    ].iloc[0]

    comparison_results.append({
        "note_id": row["note_id"],
        "prompt_version": "V1 - Simple",
        "root_cause": v1_data.get("root_cause", ""),
        "confidence": v1_data.get("confidence", ""),
        "valid_root_cause": (
            v1_data.get("root_cause", "") in ALLOWED_ROOT_CAUSES
        )
    })

    comparison_results.append({
        "note_id": row["note_id"],
        "prompt_version": "V2 - CRAFT",
        "root_cause": v2_data["root_cause"],
        "confidence": v2_data["confidence"],
        "valid_root_cause": (
            v2_data["root_cause"] in ALLOWED_ROOT_CAUSES
        )
    })

    time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)

print(comparison_df.to_string(index=False))
print("\nTotal API calls:", api_call_count)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 8h11m57.60478242s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.go

In [37]:
import os

print("appointments_clean.csv exists:",
      os.path.exists("appointments_clean.csv"))

appointments_clean.csv exists: False


In [38]:
appointments_df = pd.read_csv("appointments_clean.csv")

merged_df = analysed_df.merge(
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("Merged records:", len(merged_df))
print("\nRoot cause vs reminder sent:")
crosstab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print(crosstab)

Merged records: 10

Root cause vs reminder sent:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1


In [39]:
analysed_df.to_csv("notes_analysed.csv", index=False)

print("Final notes_analysed.csv saved!")
print("Rows:", len(analysed_df))
print("Columns:", list(analysed_df.columns))

Final notes_analysed.csv saved!
Rows: 10
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']
